[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abpandey-dev/data-engineering-from-scratch-to-advanced/blob/main/notebooks/topics/14_kafka.ipynb)

*Data Engineering from Scratch to Advanced · Notebook 14 of 21*

# Kafka & Messaging

The backbone of streaming: logs, topics and partitions, keys and ordering, consumer groups, offsets and lag, delivery guarantees, replication, retention and compaction, schemas. You'll build a tiny Kafka in Python.

### 👩‍🏫 From your teacher

**The picture to keep in your head:** Kafka is a notebook you only ever add to. Each reader keeps its own bookmark, and can flip back.

**What interviewers really test here:** Partitions and ordering, consumer groups, lag, delivery guarantees and replication.

### How to use this notebook

1. **Run the setup cell** just below.
2. **Read each lesson slowly.** Work the numbers on paper before reading on.
3. **Run and then change the examples.** Breaking code on purpose is how you learn.
4. **Try each practice question before opening an answer.** If you're stuck, open one hint at a time.
5. **Say your interview answers out loud.** Knowing something and explaining it are different skills.

### Your roadmap

- [ ] 1. Kafka is a log *(Basics)*
- [ ] 2. Consumer groups, offsets and lag *(Core)*
- [ ] 3. Delivery guarantees in practice *(Core)*
- [ ] 4. Replication, retention and schemas *(Advanced)*
- [ ] 20 practice questions
- [ ] Interview questions, out loud

In [1]:
# ▶ Run this cell first. It prepares the notebook; you don't need to read it.
import contextlib, copy, io, math, sys
import numpy as np, matplotlib.pyplot as plt
%matplotlib inline
import math
import numpy as np
import matplotlib.pyplot as plt
PLAIN = True

class ListNode:
    def __init__(self, val=0, next=None):
        self.val, self.next = val, next

class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def mk_list(vals, pos=-1):
    dummy = cur = ListNode()
    nodes = []
    for v in vals:
        cur.next = ListNode(v)
        cur = cur.next
        nodes.append(cur)
    if pos >= 0 and nodes:
        nodes[-1].next = nodes[pos]
    return dummy.next

def list_arr(head):
    out, seen = [], set()
    while head and id(head) not in seen:
        seen.add(id(head))
        out.append(head.val)
        head = head.next
    return out

def mk_tree(vals):
    if not vals or vals[0] is None:
        return None
    root = TreeNode(vals[0])
    queue, i = [root], 1
    while queue and i < len(vals):
        node = queue.pop(0)
        for side in ("left", "right"):
            if i < len(vals) and vals[i] is not None:
                child = TreeNode(vals[i])
                setattr(node, side, child)
                queue.append(child)
            i += 1
    return root

def tree_arr(root):
    out, queue = [], [root]
    while queue:
        node = queue.pop(0)
        if node:
            out.append(node.val)
            queue += [node.left, node.right]
        else:
            out.append(None)
    while out and out[-1] is None:
        out.pop()
    return out

def find(root, val):
    if not root or root.val == val:
        return root
    return find(root.left, val) or find(root.right, val)

ListNode.__repr__ = lambda self: f"ListNode{list_arr(self)}"
TreeNode.__repr__ = lambda self: f"TreeNode{tree_arr(self)}"

def plain(x, digits=4):
    """NumPy / pandas results as plain Python values, so they compare with a test's expected answer."""
    np = sys.modules.get("numpy")
    pd = sys.modules.get("pandas")
    if pd is not None:
        if isinstance(x, pd.DataFrame):
            return {str(c): plain(x[c].tolist(), digits) for c in x.columns}
        if isinstance(x, pd.Series):
            return {plain(k, digits): plain(v, digits) for k, v in x.items()}
        if isinstance(x, pd.Index):
            return plain(x.tolist(), digits)
        if x is pd.NaT:
            return None
        if isinstance(x, pd.Timestamp):
            return str(x.date()) if x == x.normalize() else str(x)
    if np is not None:
        if isinstance(x, np.ndarray):
            return plain(x.tolist(), digits)
        if isinstance(x, np.generic):
            return plain(x.item(), digits)
    if isinstance(x, float):
        return None if x != x else round(x, digits) + 0.0
    if isinstance(x, list):
        return [plain(v, digits) for v in x]
    if isinstance(x, tuple):
        return tuple(plain(v, digits) for v in x)
    if isinstance(x, dict):
        return {plain(k, digits): plain(v, digits) for k, v in x.items()}
    if isinstance(x, (set, frozenset)):
        return {plain(v, digits) for v in x}
    return x


def check(fn, tests, plain_compare=None, listify=False, design=False, prep=None, post=None,
          norm=None, verify=None, runner=None, inplace=None):
    """Run your code against the tests and show which pass."""
    use_plain = PLAIN if plain_compare is None else plain_compare
    passed, problems = 0, []
    for args, want in tests:
        try:
            a = copy.deepcopy(args)
            if design:
                obj = fn(*a[0])
                got = [getattr(obj, m)(*p) for m, p in a[1]]
            elif runner is not None:
                got = runner(fn, a)
            else:
                if prep is not None:
                    a = prep(a)
                with contextlib.redirect_stdout(io.StringIO()):
                    got = fn(*a)
                if inplace is not None:
                    got = a[inplace]
                if listify:
                    got = list(got)
                if post is not None:
                    got = post(got)
            if verify is not None:
                ok = bool(verify(copy.deepcopy(args), got))
            elif norm is not None:
                ok = norm(copy.deepcopy(got)) == norm(copy.deepcopy(want))
            else:
                ok = (plain(got), plain(want)) if use_plain else (got, want)
                ok = ok[0] == ok[1]
            ok = ok if isinstance(ok, bool) else False
        except Exception as e:  # your code raised an error: report it, don't stop the notebook
            ok, got = False, f"{type(e).__name__}: {e}"
        passed += ok
        if not ok:
            problems.append((args, want, got))
    plt.close("all")
    if passed == len(tests):
        print(f"✅ All {len(tests)} tests passed. Well done! Now compare with the answers below.")
    else:
        print(f"❌ {passed}/{len(tests)} tests passed. First one to fix:")
        args, want, got = problems[0]
        print("   input:   ", repr(args)[:300])
        print("   expected:", repr(want)[:300])
        print("   you got: ", repr(got)[:300])
    return None

print("Ready! Scroll down and start learning.")

Ready! Scroll down and start learning.


---
# Part 1: The theory in one place

Read this once for the big picture. It's fine if it doesn't all stick yet: the lessons will make it concrete, and you can come back to revise.

### Kafka vs a queue

|   | Kafka | Traditional queue (e.g. SQS, RabbitMQ) |
| --- | --- | --- |
| After reading | messages stay until retention | usually removed |
| Replay | yes | generally no |
| Order | per partition | varies |
| Throughput | very high | moderate |
| Best for | event streams, many readers | task queues |

---
# Part 2: Lessons, step by step

## Lesson 1: Kafka is a log

*Level: Basics*

Service A wants to tell services B, C and D about each order. Direct calls tangle everything together, and if D is down, data's lost. **Kafka** sits in the middle: A writes events, and each reader goes at its own pace, independently.



**The core idea is an append-only log.** New messages go at the end, and each gets a number, its **offset**. Reading doesn't delete anything.



```text
offset:   0    1    2    3    4
        [a]  [b]  [c]  [d]  [e]  ← new writes
                   ↑         ↑
             consumer A   consumer B (each tracks its own place)
```



Because a consumer is just a position in a log, you can **replay history** by moving back. That's very different from a traditional queue, which removes messages once read.



| Term | Meaning |
| --- | --- |
| Topic | a named stream, such as "orders" |
| Partition | one log; a topic has many, for scale |
| Broker | a Kafka server |
| Producer / consumer | writes / reads |
| Offset | a position in a partition |



**Order is only guaranteed within a partition.** With a key, all records for that key go to one partition, and stay in order. So use the order id or user id as the key for events that must stay in sequence.



Choose keys with care. Using "country" could send 82% of traffic to one partition, and overload it.



The code examples here need a running Kafka, so questions model the behaviour in Python.



```python
from confluent_kafka import Producer
p = Producer({"bootstrap.servers": "localhost:9092", "enable.idempotence": True, "acks": "all"})
p.produce("orders", key=str(order_id), value=payload)
p.flush()
```

**✏️ Your turn:** you now have what you need for [Q1](#q1), [Q2](#q2), [Q7](#q7), [Q6](#q6). Try one now, or carry on and do them in Part 3.

## Lesson 2: Consumer groups, offsets and lag

*Level: Core*

**A consumer group shares the work.** Each partition goes to exactly one member, so partitions cap your parallelism: 4 partitions with 2 consumers gives each 2. With 3 consumers on 2 partitions, one sits idle.



**Different groups are independent,** so analytics and fraud detection can read the same topic at their own speeds.



**Lag** is how far behind a consumer is: latest offset minus committed offset. It's the most important Kafka metric to alert on. Partition 0 at 100 with committed 90 has a lag of 10.



**Will it catch up?** With lag 1,000, producing 100/s and consuming 200/s, you gain 100/s, so 10 seconds. If you consume slower than you produce, it never catches up, so add consumers (up to the partition count) or speed up processing.



**Rebalancing.** When a member joins or dies, partitions are reassigned. With the old "eager" approach, everyone stops. Going from 2 consumers to 3 with 8 partitions, sticky or cooperative assignment moves only 2 partitions rather than all 8.



**Sizing partitions.** Target 100 MB/s, with 10 MB/s per partition on produce and 20 on consume: at least 10. You can add partitions later, but this changes which partition a key maps to, so plan ahead.



**Disk:** 50 MB/s, 7 days, replication 3 takes about 91 TB.

**✏️ Your turn:** you now have what you need for [Q8](#q8), [Q3](#q3), [Q9](#q9), [Q4](#q4), [Q17](#q17), [Q19](#q19). Try one now, or carry on and do them in Part 3.

## Lesson 3: Delivery guarantees in practice

*Level: Core*

The order of two steps decides what a crash does:



| Consumer does | Crash between them | Result |
| --- | --- | --- |
| Commit, then process | message never processed | **at most once**: can lose |
| Process, then commit | message processed again | **at least once**: can repeat |



**At-least-once by hand.** Messages a to e, committing every 2, crashing after c. The last commit is at 2, so we restart there: a, b, c, **c**, d, e. One duplicate.



**The practical answer: at-least-once plus idempotent processing.** Applying (1, 10), (2, 5), (1, 10), (3, 1), ignoring repeated ids, gives 16 rather than 26. Store the processed id in the **same transaction** as the result, or use upserts.



**Producer duplicates.** A producer that doesn't get an acknowledgement retries, and might write twice. With `enable.idempotence=True`, brokers deduplicate using a producer id and sequence number. It also prevents reordering on retry, where b1 fails, b2 succeeds, and b1 lands after it.



**Kafka transactions** give exactly-once between Kafka topics, by writing output and offsets atomically. To an external system, you still need idempotence.



**The outbox pattern.** Writing to a database and to Kafka separately can fail in between: 3 rows but 2 events. Instead, write the row and an "outbox" event in one database transaction, and let a relay or CDC publish it. Database and stream can't disagree.

**✏️ Your turn:** you now have what you need for [Q10](#q10), [Q11](#q11), [Q12](#q12), [Q20](#q20). Try one now, or carry on and do them in Part 3.

## Lesson 4: Replication, retention and schemas

*Level: Advanced*

**Replication.** Each partition has one leader and several followers. **In-sync replicas (ISR)** are those keeping up.



`acks=all` waits for all in-sync replicas. With `min.insync.replicas=2` and replication 3, you can lose a broker with no data loss. If too few remain, writes are refused: consistency over availability.



**High watermark.** Consumers can read only what's on all in-sync replicas. With offsets 100, 95, 98, they read up to 95, so a new leader can't lose what anyone has read.



**Retention.** Kafka deletes old segments by time or size. **Log compaction** instead keeps the last value per key, like a table: a:1, b:2, a:3, b:None, c:9 keeps a:3 and c:9. A None value, called a tombstone, means delete. It's ideal for "current state" topics.



**Batching.** Producers wait up to `linger.ms` to fill a batch, so you trade some latency for much better throughput.



**Schemas.** A consumer breaks when a producer changes a message. A **schema registry** checks compatibility. Backward compatible means new readers can read old data, so added fields need defaults.



| Change | Backward compatible? |
| --- | --- |
| Add a field with a default | yes |
| Add a required field | no |
| Remove a field | yes for backward, but check forward |

**✏️ Your turn:** you now have what you need for [Q13](#q13), [Q14](#q14), [Q15](#q15), [Q16](#q16), [Q18](#q18), [Q5](#q5). Try one now, or carry on and do them in Part 3.

---
# Part 3: Practice like an interview

There are 20 questions here, easiest first.

**Work the way you would in a real interview:** restate the problem, try an example by hand, say the simple approach and its cost, improve it, then write the code. Test it with the cell. Only open an answer once you've made a real attempt, because struggling is where the learning happens.

<a id="q1"></a>
### Q1. Append to a Log

🟢 Warm-up · *The core data structure*

Append messages to a list log and return their offsets, then read from a given offset.

**Examples**

- `messages = ['a', 'b', 'c'], start = 1` → `([0, 1, 2], ['b', 'c'])`

In [ ]:
def log_append(messages, start):
    """
    Append messages to a list log and return their offsets, then read from a
    given offset.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(log_append, [
    ((['a', 'b', 'c'], 1), ([0, 1, 2], ['b', 'c'])),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

A loop for appends.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`range` and slicing.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** A loop for appends.

```python
def log_append(messages, start):
    log, offsets = [], []
    for m in messages:
        offsets.append(len(log))
        log.append(m)
    return (offsets, log[start:])
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** `range` and slicing.

```python
def log_append(messages, start):
    return (list(range(len(messages))), messages[start:])
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: A loop for appends. That costs O(n) time and O(n) space."

"I can do better: `range` and slicing. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/kafka/append-to-a-log)

<a id="q2"></a>
### Q2. Choose a Partition by Key

🟢 Warm-up · *Same key, same partition*

Send (key, value) records to partitions with sum(codes) % n. Return partition contents. Order within each key is preserved.

**Examples**

- `records = [('a', 1), ('b', 2), ('a', 3)], n = 2` → `[[2], [1, 3]]`

In [ ]:
def produce(records, n):
    """
    Send (key, value) records to partitions with sum(codes) % n. Return
    partition contents. Order within each key is preserved.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(produce, [
    (([('a', 1), ('b', 2), ('a', 3)], 2), [[2], [1, 3]]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

A list of lists.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Same, compactly.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** A list of lists.

```python
def produce(records, n):
    parts = []
    for _ in range(n):
        parts.append([])
    for k, v in records:
        total = 0
        for c in k:
            total += ord(c)
        parts[total % n].append(v)
    return parts
```

**Cost:** O(k) time, O(k) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Same, compactly.

```python
def produce(records, n):
    parts = [[] for _ in range(n)]
    for k, v in records:
        parts[sum(map(ord, k)) % n].append(v)
    return parts
```

**Cost:** O(k) time, O(k) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: A list of lists. That costs O(k) time and O(k) space."

"I can do better: Same, compactly. That's O(k) time and O(k) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/kafka/choose-a-partition-by-key)

<a id="q3"></a>
### Q3. Consumer Lag

🟢 Warm-up · *The number to alert on*

Given latest offsets and committed offsets per partition, return lag per partition and in total.

**Examples**

- `latest = {0: 100, 1: 50}, committed = {0: 90, 1: 50}` → `({0: 10, 1: 0}, 10)`

In [ ]:
def consumer_lag(latest, committed):
    """
    Given latest offsets and committed offsets per partition, return lag per
    partition and in total.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(consumer_lag, [
    (({0: 100, 1: 50}, {0: 90, 1: 50}), ({0: 10, 1: 0}, 10)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

A loop.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A comprehension.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** A loop.

```python
def consumer_lag(latest, committed):
    lags, total = {}, 0
    for p in latest:
        lag = latest[p] - committed.get(p, 0)
        lags[p] = lag
        total += lag
    return (lags, total)
```

**Cost:** O(p) time, O(p) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A comprehension.

```python
def consumer_lag(latest, committed):
    lags = {p: v - committed.get(p, 0) for p, v in latest.items()}
    return (lags, sum(lags.values()))
```

**Cost:** O(p) time, O(p) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: A loop. That costs O(p) time and O(p) space."

"I can do better: A comprehension. That's O(p) time and O(p) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/kafka/consumer-lag)

<a id="q4"></a>
### Q4. Partitions Needed

🟢 Warm-up · *Sizing a topic*

Given target throughput and per-partition throughput for producers and consumers, return the partitions needed: the larger of the two, rounded up.

**Examples**

- `target = 100, produce_each = 10, consume_each = 20` → `10`
- `target = 100, produce_each = 50, consume_each = 5` → `20`

In [ ]:
def partitions_needed(target, produce_each, consume_each):
    """
    Given target throughput and per-partition throughput for producers and
    consumers, return the partitions needed: the larger of the two, rounded up.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(partitions_needed, [
    ((100, 10, 20), 10),
    ((100, 50, 5), 20),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Divide and compare.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`ceil(max(...))`.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Divide and compare.

```python
def partitions_needed(target, produce_each, consume_each):
    a = -(-target // produce_each)
    b = -(-target // consume_each)
    return a if a > b else b
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** `ceil(max(...))`.

```python
def partitions_needed(target, produce_each, consume_each):
    import math
    return math.ceil(max(target / produce_each, target / consume_each))
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Divide and compare. That costs O(1) time and O(1) space."

"I can do better: `ceil(max(...))`. That's O(1) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/kafka/partitions-needed)

<a id="q5"></a>
### Q5. Batching Trade-off

🟢 Warm-up · *Latency vs throughput*

A producer sends when a batch reaches size or after linger ms. At rate messages per ms, return the time to fill a batch rounded to 2 decimals, and the actual wait.

**Examples**

- `batch = 1000, rate = 5, linger = 10` → `(200.0, 10)`
- `batch = 100, rate = 50, linger = 10` → `(2.0, 2.0)`

In [ ]:
def batching(batch, rate, linger):
    """
    A producer sends when a batch reaches size or after linger ms. At rate
    messages per ms, return the time to fill a batch rounded to 2 decimals, and
    the actual wait.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(batching, [
    ((1000, 5, 10), (200.0, 10)),
    ((100, 50, 10), (2.0, 2.0)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Divide and compare.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`min`.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Divide and compare.

```python
def batching(batch, rate, linger):
    fill = batch / rate
    wait = fill if fill < linger else linger
    return (round(fill, 2), wait)
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** `min`.

```python
def batching(batch, rate, linger):
    return (round(batch / rate, 2), min(batch / rate, linger))
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Divide and compare. That costs O(1) time and O(1) space."

"I can do better: `min`. That's O(1) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/kafka/batching-trade-off)

<a id="q6"></a>
### Q6. Find the Hot Partition

🟢 Warm-up · *A bad key choice*

Given partition message counts, return (largest partition, share of all traffic rounded to 2 decimals). Choosing a key like country can overload one partition.

**Examples**

- `counts = [100, 900, 50, 50]` → `(1, 0.82)`

In [ ]:
def hot_partition(counts):
    """
    Given partition message counts, return (largest partition, share of all
    traffic rounded to 2 decimals). Choosing a key like country can overload one
    partition.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(hot_partition, [
    (([100, 900, 50, 50],), (1, 0.82)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

A loop.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`max`.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** A loop.

```python
def hot_partition(counts):
    top, total = 0, 0
    for i, c in enumerate(counts):
        total += c
        if c > counts[top]:
            top = i
    return (top, round(counts[top] / total, 2))
```

**Cost:** O(p) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** `max`.

```python
def hot_partition(counts):
    top = counts.index(max(counts))
    return (top, round(max(counts) / sum(counts), 2))
```

**Cost:** O(p) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: A loop. That costs O(p) time and O(1) space."

"I can do better: `max`. That's O(p) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/kafka/find-the-hot-partition)

<a id="q7"></a>
### Q7. Does Order Hold?

🟡 Interview standard · *Only within a partition*

A consumer may read partitions in any interleaving. Given partition contents and an interleaving list of partition ids, return the sequence read.

**Examples**

- `parts = [['a1', 'a2'], ['b1']], order = [1, 0, 0]` → `['b1', 'a1', 'a2']`

In [ ]:
def interleave(parts, order):
    """
    A consumer may read partitions in any interleaving. Given partition contents
    and an interleaving list of partition ids, return the sequence read.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(interleave, [
    (([['a1', 'a2'], ['b1']], [1, 0, 0]), ['b1', 'a1', 'a2']),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Track an offset per partition with a list.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Iterators, one per partition.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Track an offset per partition with a list.

```python
def interleave(parts, order):
    offsets = [0] * len(parts)
    out = []
    for p in order:
        out.append(parts[p][offsets[p]])
        offsets[p] += 1
    return out
```

**Cost:** O(n) time, O(p) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Iterators, one per partition.

```python
def interleave(parts, order):
    its = [iter(p) for p in parts]
    return [next(its[p]) for p in order]
```

**Cost:** O(n) time, O(p) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Track an offset per partition with a list. That costs O(n) time and O(p) space."

"I can do better: Iterators, one per partition. That's O(n) time and O(p) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/kafka/does-order-hold)

<a id="q8"></a>
### Q8. Consumer Group Assignment

🟡 Interview standard · *Each partition goes to one member*

Assign partitions 0..p−1 to sorted consumers in round-robin order. Return {consumer: partitions}. Extra consumers sit idle.

**Examples**

- `p = 4, consumers = ['c1', 'c2']` → `{'c1': [0, 2], 'c2': [1, 3]}`
- `p = 2, consumers = ['c1', 'c2', 'c3']` → `{'c1': [0], 'c2': [1], 'c3': []}`

In [ ]:
def assign(p, consumers):
    """
    Assign partitions 0..p−1 to sorted consumers in round-robin order. Return
    {consumer: partitions}. Extra consumers sit idle.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(assign, [
    ((4, ['c1', 'c2']), {'c1': [0, 2], 'c2': [1, 3]}),
    ((2, ['c1', 'c2', 'c3']), {'c1': [0], 'c2': [1], 'c3': []}),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

A loop with an index.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Slicing with a step.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** A loop with an index.

```python
def assign(p, consumers):
    names = sorted(consumers)
    out = {c: [] for c in names}
    for i in range(p):
        out[names[i % len(names)]].append(i)
    return out
```

**Cost:** O(p + c) time, O(p) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Slicing with a step.

```python
def assign(p, consumers):
    names = sorted(consumers)
    return {c: list(range(i, p, len(names))) for i, c in enumerate(names)}
```

**Cost:** O(p + c) time, O(p) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: A loop with an index. That costs O(p + c) time and O(p) space."

"I can do better: Slicing with a step. That's O(p + c) time and O(p) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/kafka/consumer-group-assignment)

<a id="q9"></a>
### Q9. How Long to Catch Up?

🟡 Interview standard · *Is the consumer keeping up?*

With lag L, producing at in\_rate and consuming at out\_rate (messages per second), return seconds to catch up rounded to 1 decimal, or None if it never will.

**Examples**

- `lag = 1000, in_rate = 100, out_rate = 200` → `10.0`
- `lag = 1000, in_rate = 200, out_rate = 100` → `None`

In [ ]:
def catch_up(lag, in_rate, out_rate):
    """
    With lag L, producing at in_rate and consuming at out_rate (messages per
    second), return seconds to catch up rounded to 1 decimal, or None if it
    never will.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(catch_up, [
    ((1000, 100, 200), 10.0),
    ((1000, 200, 100), None),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

An if, then divide.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A conditional expression.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** An if, then divide.

```python
def catch_up(lag, in_rate, out_rate):
    net = out_rate - in_rate
    if net <= 0:
        return None
    return round(lag / net, 1)
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A conditional expression.

```python
def catch_up(lag, in_rate, out_rate):
    return round(lag / (out_rate - in_rate), 1) if out_rate > in_rate else None
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: An if, then divide. That costs O(1) time and O(1) space."

"I can do better: A conditional expression. That's O(1) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/kafka/how-long-to-catch-up)

<a id="q10"></a>
### Q10. Commit After Processing

🟡 Interview standard · *At-least-once*

Process messages, committing every c. A crash happens after processing crash\_after. On restart, processing resumes from the last commit. Return (all processed values in order, duplicates).

**Examples**

- `messages = ['a', 'b', 'c', 'd', 'e'], c = 2, crash_after = 3` → `(['a', 'b', 'c', 'c', 'd', 'e'], 1)`

In [ ]:
def at_least_once(messages, c, crash_after):
    """
    Process messages, committing every c. A crash happens after processing
    crash_after. On restart, processing resumes from the last commit. Return
    (all processed values in order, duplicates).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(at_least_once, [
    ((['a', 'b', 'c', 'd', 'e'], 2, 3), (['a', 'b', 'c', 'c', 'd', 'e'], 1)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Simulate with loops.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Compute the commit point directly.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Simulate with loops.

```python
def at_least_once(messages, c, crash_after):
    done, committed = [], 0
    for i in range(crash_after):
        done.append(messages[i])
        if (i + 1) % c == 0:
            committed = i + 1
    for i in range(committed, len(messages)):
        done.append(messages[i])
    return (done, crash_after - committed)
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Compute the commit point directly.

```python
def at_least_once(messages, c, crash_after):
    committed = crash_after // c * c
    return (messages[:crash_after] + messages[committed:], crash_after - committed)
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Simulate with loops. That costs O(n) time and O(n) space."

"I can do better: Compute the commit point directly. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/kafka/commit-after-processing)

<a id="q11"></a>
### Q11. Idempotent Consumer

🟡 Interview standard · *Exactly-once in effect*

Apply messages (id, delta) to a total, ignoring ids already seen. Duplicates from retries don't change the result. Return (total, applied).

**Examples**

- `messages = [(1, 10), (2, 5), (1, 10), (3, 1)]` → `(16, 3)`

In [ ]:
def idempotent_consumer(messages):
    """
    Apply messages (id, delta) to a total, ignoring ids already seen. Duplicates
    from retries don't change the result. Return (total, applied).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(idempotent_consumer, [
    (([(1, 10), (2, 5), (1, 10), (3, 1)],), (16, 3)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

A list of seen ids.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A set of ids. In production, store them in the same transaction as the result.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** A list of seen ids.

```python
def idempotent_consumer(messages):
    seen, total, applied = [], 0, 0
    for i, delta in messages:
        if i not in seen:
            seen.append(i)
            total += delta
            applied += 1
    return (total, applied)
```

**Cost:** O(n²) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A set of ids. In production, store them in the same transaction as the result.

```python
def idempotent_consumer(messages):
    unique = dict(messages)
    return (sum(unique.values()), len(unique))
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: A list of seen ids. That costs O(n²) time and O(n) space."

"I can do better: A set of ids. In production, store them in the same transaction as the result. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/kafka/idempotent-consumer)

<a id="q12"></a>
### Q12. Producer Retries Reorder

🟡 Interview standard · *max.in.flight*

Two batches are in flight; batch 1 fails, and is retried after batch 2 succeeds. Return the log order without idempotence, and with an idempotent producer preserving order.

**Examples**

- `batches = ['b1', 'b2']` → `(['b2', 'b1'], ['b1', 'b2'])`

In [ ]:
def retry_order(batches):
    """
    Two batches are in flight; batch 1 fails, and is retried after batch 2
    succeeds. Return the log order without idempotence, and with an idempotent
    producer preserving order.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(retry_order, [
    ((['b1', 'b2'],), (['b2', 'b1'], ['b1', 'b2'])),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Simulate each.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Slicing.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Simulate each.

```python
def retry_order(batches):
    log = []
    log.append(batches[1])
    log.append(batches[0])
    return (log, list(batches))
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Slicing.

```python
def retry_order(batches):
    return (batches[::-1], batches)
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Simulate each. That costs O(n) time and O(n) space."

"I can do better: Slicing. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/kafka/producer-retries-reorder)

<a id="q13"></a>
### Q13. Log Compaction

🟡 Interview standard · *Keep the latest per key*

Keep only the last record per key, and drop keys whose last value is None (a tombstone). Return remaining (offset, key, value), in offset order.

**Examples**

- `records = [('a', 1), ('b', 2), ('a', 3), ('b', None), ('c', 9)]` → `[[2, 'a', 3], [4, 'c', 9]]`

In [ ]:
def compact(records):
    """
    Keep only the last record per key, and drop keys whose last value is None (a
    tombstone). Return remaining (offset, key, value), in offset order.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(compact, [
    (([('a', 1), ('b', 2), ('a', 3), ('b', None), ('c', 9)],), [[2, 'a', 3], [4, 'c', 9]]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

For each record, check for a later one.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A dictionary of the last offset.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** For each record, check for a later one.

```python
def compact(records):
    out = []
    for i, (k, v) in enumerate(records):
        later = any(records[j][0] == k for j in range(i + 1, len(records)))
        if not later and v is not None:
            out.append([i, k, v])
    return out
```

**Cost:** O(n²) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A dictionary of the last offset.

```python
def compact(records):
    last = {k: i for i, (k, _) in enumerate(records)}
    return [[i, k, records[i][1]] for k, i in sorted(last.items(), key=lambda x: x[1]) if records[i][1] is not None]
```

**Cost:** O(n) time, O(k) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: For each record, check for a later one. That costs O(n²) time and O(n) space."

"I can do better: A dictionary of the last offset. That's O(n) time and O(k) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/kafka/log-compaction)

<a id="q14"></a>
### Q14. Retention by Time and Size

🟡 Interview standard · *Old data goes away*

Segments are (newest time, MB), oldest first. Delete segments older than retention (now − newest &gt; hours), and then oldest ones until total ≤ max\_mb. Return the remaining count and MB.

**Examples**

- `segments = [(1, 100), (50, 100), (90, 100), (99, 100)], now = 100, hours = 60, max_mb = 250` → `(2, 200)`

In [ ]:
def retention(segments, now, hours, max_mb):
    """
    Segments are (newest time, MB), oldest first. Delete segments older than
    retention (now − newest > hours), and then oldest ones until total ≤ max_mb.
    Return the remaining count and MB.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(retention, [
    (([(1, 100), (50, 100), (90, 100), (99, 100)], 100, 60, 250), (2, 200)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

A loop for each rule.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A filter, then trim.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** A loop for each rule.

```python
def retention(segments, now, hours, max_mb):
    kept = []
    for t, mb in segments:
        if now - t <= hours:
            kept.append((t, mb))
    while sum(m for _, m in kept) > max_mb:
        kept.pop(0)
    return (len(kept), sum(m for _, m in kept))
```

**Cost:** O(s) time, O(s) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A filter, then trim.

```python
def retention(segments, now, hours, max_mb):
    from collections import deque
    kept = deque(s for s in segments if now - s[0] <= hours)
    total = sum(m for _, m in kept)
    while total > max_mb:
        total -= kept.popleft()[1]
    return (len(kept), total)
```

**Cost:** O(s) time, O(s) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: A loop for each rule. That costs O(s) time and O(s) space."

"I can do better: A filter, then trim. That's O(s) time and O(s) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/kafka/retention-by-time-and-size)

<a id="q15"></a>
### Q15. In-Sync Replicas

🟡 Interview standard · *Durability trade-off*

A replica is in sync if within max\_lag of the leader's offset. Given offsets, return (ISR), and whether a write with acks=all can proceed under min\_isr.

**Examples**

- `offsets = {'r1': 100, 'r2': 99, 'r3': 50}, leader = 'r1', max_lag = 5, min_isr = 2` → `(['r1', 'r2'], True)`
- `offsets = {'r1': 100, 'r2': 50}, leader = 'r1', max_lag = 5, min_isr = 2` → `(['r1'], False)`

In [ ]:
def isr(offsets, leader, max_lag, min_isr):
    """
    A replica is in sync if within max_lag of the leader's offset. Given
    offsets, return (ISR), and whether a write with acks=all can proceed under
    min_isr.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(isr, [
    (({'r1': 100, 'r2': 99, 'r3': 50}, 'r1', 5, 2), (['r1', 'r2'], True)),
    (({'r1': 100, 'r2': 50}, 'r1', 5, 2), (['r1'], False)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

A loop.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A sorted comprehension.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** A loop.

```python
def isr(offsets, leader, max_lag, min_isr):
    top = offsets[leader]
    out = []
    for r, o in offsets.items():
        if top - o <= max_lag:
            out.append(r)
    return (sorted(out), len(out) >= min_isr)
```

**Cost:** O(r log r) time, O(r) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A sorted comprehension.

```python
def isr(offsets, leader, max_lag, min_isr):
    s = sorted(r for r, o in offsets.items() if offsets[leader] - o <= max_lag)
    return (s, len(s) >= min_isr)
```

**Cost:** O(r log r) time, O(r) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: A loop. That costs O(r log r) time and O(r) space."

"I can do better: A sorted comprehension. That's O(r log r) time and O(r) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/kafka/in-sync-replicas)

<a id="q16"></a>
### Q16. High Watermark

🟡 Interview standard · *What consumers can read*

Consumers only see messages replicated to all in-sync replicas: the minimum of their offsets. Return it, and how many records the leader has that aren't visible.

**Examples**

- `isr_offsets = [100, 95, 98]` → `(95, 5)`

In [ ]:
def high_watermark(isr_offsets):
    """
    Consumers only see messages replicated to all in-sync replicas: the minimum
    of their offsets. Return it, and how many records the leader has that aren't
    visible.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(high_watermark, [
    (([100, 95, 98],), (95, 5)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

A loop for the minimum.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`min` and `max`.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** A loop for the minimum.

```python
def high_watermark(isr_offsets):
    low = isr_offsets[0]
    for o in isr_offsets:
        if o < low:
            low = o
    return (low, isr_offsets[0] - low)
```

**Cost:** O(r) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** `min` and `max`.

```python
def high_watermark(isr_offsets):
    return (min(isr_offsets), max(isr_offsets) - min(isr_offsets))
```

**Cost:** O(r) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: A loop for the minimum. That costs O(r) time and O(1) space."

"I can do better: `min` and `max`. That's O(r) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/kafka/high-watermark)

<a id="q17"></a>
### Q17. Rebalance Movement

🟡 Interview standard · *Eager vs sticky*

Given old assignment and new round-robin (sorted) after a member joins, return partitions that moved with eager (all revoke, so all) and with sticky assignment that keeps as many as possible. Sticky here keeps a partition if its old owner is below its target share.

**Examples**

- `old = {'c1': [0, 1, 2, 3], 'c2': [4, 5, 6, 7]}, members = ['c1', 'c2', 'c3']` → `(8, 2)`

In [ ]:
def rebalance(old, members):
    """
    Given old assignment and new round-robin (sorted) after a member joins,
    return partitions that moved with eager (all revoke, so all) and with sticky
    assignment that keeps as many as possible. Sticky here keeps a partition if
    its old owner is below its target share.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(rebalance, [
    (({'c1': [0, 1, 2, 3], 'c2': [4, 5, 6, 7]}, ['c1', 'c2', 'c3']), (8, 2)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Simulate with loops.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Sticky only moves the excess.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Simulate with loops.

```python
def rebalance(old, members):
    all_parts = sum(old.values(), [])
    target = len(all_parts) // len(members)
    extra = len(all_parts) % len(members)
    moved = 0
    for i, m in enumerate(sorted(members)):
        share = target + (1 if i < extra else 0)
        have = len(old.get(m, []))
        if have > share:
            moved += have - share
    return (len(all_parts), moved)
```

**Cost:** O(p) time, O(p) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Sticky only moves the excess.

```python
def rebalance(old, members):
    total = sum(map(len, old.values()))
    q, r = divmod(total, len(members))
    return (total, sum(max(0, len(old.get(m, [])) - (q + (i < r))) for i, m in enumerate(sorted(members))))
```

**Cost:** O(c) time, O(c) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Simulate with loops. That costs O(p) time and O(p) space."

"I can do better: Sticky only moves the excess. That's O(c) time and O(c) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/kafka/rebalance-movement)

<a id="q18"></a>
### Q18. Schema Compatibility

🟡 Interview standard · *Don't break consumers*

Backward compatible: new consumers can read old data, so any added field needs a default, and removing is fine. Fields are {name: has\_default}. Return (compatible, problem fields sorted).

**Examples**

- `old = {'id': False}, new = {'id': False, 'x': True}` → `(True, [])`
- `old = {'id': False}, new = {'id': False, 'x': False}` → `(False, ['x'])`

In [ ]:
def backward_compatible(old, new):
    """
    Backward compatible: new consumers can read old data, so any added field
    needs a default, and removing is fine. Fields are {name: has_default}.
    Return (compatible, problem fields sorted).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(backward_compatible, [
    (({'id': False}, {'id': False, 'x': True}), (True, [])),
    (({'id': False}, {'id': False, 'x': False}), (False, ['x'])),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

A loop.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A comprehension.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** A loop.

```python
def backward_compatible(old, new):
    problems = []
    for name, has_default in new.items():
        if name not in old and not has_default:
            problems.append(name)
    return (not problems, sorted(problems))
```

**Cost:** O(f) time, O(f) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A comprehension.

```python
def backward_compatible(old, new):
    bad = sorted(n for n, d in new.items() if n not in old and not d)
    return (not bad, bad)
```

**Cost:** O(f log f) time, O(f) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: A loop. That costs O(f) time and O(f) space."

"I can do better: A comprehension. That's O(f log f) time and O(f) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/kafka/schema-compatibility)

<a id="q19"></a>
### Q19. Kafka Disk Sizing

🟡 Interview standard · *Capacity planning*

Return TB needed for rate MB/s, retention days and replication factor. TB = 10⁶ MB. Round to 2 decimals.

**Examples**

- `mb_s = 50, days = 7, replicas = 3` → `90.72`

In [ ]:
def kafka_disk(mb_s, days, replicas):
    """
    Return TB needed for rate MB/s, retention days and replication factor. TB =
    10⁶ MB. Round to 2 decimals.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(kafka_disk, [
    ((50, 7, 3), 90.72),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Step by step.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

One line.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Step by step.

```python
def kafka_disk(mb_s, days, replicas):
    per_day = mb_s * 86400
    total = per_day * days * replicas
    return round(total / 1e6, 2)
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** One line.

```python
def kafka_disk(mb_s, days, replicas):
    return round(mb_s * 86400 * days * replicas / 1e6, 2)
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Step by step. That costs O(1) time and O(1) space."

"I can do better: One line. That's O(1) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/kafka/kafka-disk-sizing)

<a id="q20"></a>
### Q20. Outbox Pattern

🔴 Stretch · *Database and Kafka together*

Writing to a database and Kafka separately can fail halfway. Instead, write the row and an outbox event in one transaction, and a relay publishes them. Simulate: for each order, if crash is at 'after\_db', compare which rows and events exist with dual writes and with the outbox. Return ((dual rows, dual events), (outbox rows, outbox events)).

**Examples**

- `orders = 3` → `((3, 2), (3, 3))`

In [ ]:
def outbox(orders):
    """
    Writing to a database and Kafka separately can fail halfway. Instead, write
    the row and an outbox event in one transaction, and a relay publishes them.
    Simulate: for each order, if crash is at 'after_db', compare which rows and
    events exist with dual writes and with the outbox. Return ((dual rows, dual
    events), (outbox rows, outbox events)).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(outbox, [
    ((3,), ((3, 2), (3, 3))),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Simulate each with lists.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Reason it out.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Simulate each with lists.

```python
def outbox(orders):
    # dual writes: the crash on the last order happens after the DB write, before Kafka
    db, kafka = [], []
    for i in range(orders):
        db.append(i)
        if i < orders - 1:
            kafka.append(i)
    # outbox: the row and event commit together, and the relay publishes them all
    rows, events = [], []
    for i in range(orders):
        rows.append(i)
        events.append(i)
    return ((len(db), len(kafka)), (len(rows), len(events)))
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Reason it out.

```python
def outbox(orders):
    return ((orders, orders - 1), (orders, orders))
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Simulate each with lists. That costs O(n) time and O(n) space."

"I can do better: Reason it out. That's O(1) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/kafka/outbox-pattern)

---
# Part 4: Interview questions, out loud

Cover the answer, **say yours out loud**, then open it and compare. Aim for two or three clear sentences, then an example.

<details>
<summary><b>1. Why is Kafka fast?</b></summary>

Sequential disk writes, page cache use, batching and compression, and zero-copy transfer. It's just appending to files.

</details>

<details>
<summary><b>2. How does Kafka guarantee order?</b></summary>

Only within a partition. Use a key so related events go to the same partition.

</details>

<details>
<summary><b>3. What is a consumer group?</b></summary>

Consumers sharing a topic, with each partition read by one member at a time. Different groups read independently.

</details>

<details>
<summary><b>4. What is consumer lag?</b></summary>

The gap between the latest offset and the committed one. Growing lag means the consumer can't keep up.

</details>

<details>
<summary><b>5. How do you get exactly-once?</b></summary>

Idempotent producers and transactions within Kafka. To outside systems, use at-least-once delivery with idempotent writes.

</details>

<details>
<summary><b>6. What does acks=all do?</b></summary>

The leader waits for all in-sync replicas before confirming, which is the most durable option, especially with min.insync.replicas of 2 or more.

</details>

<details>
<summary><b>7. What is log compaction?</b></summary>

Keeping only the latest value per key, with tombstones for deletes, so a topic can represent current state.

</details>

<details>
<summary><b>8. Scenario: lag keeps growing. What do you do?</b></summary>

Check whether input rose or processing slowed. Add consumers up to the partition count, fix slow downstream calls, batch writes, or add partitions.

</details>

<details>
<summary><b>9. Scenario: consumers rebalance constantly. Why?</b></summary>

Processing takes longer than max.poll.interval.ms, or heartbeats are missed. Reduce batch size, speed up processing, and tune timeouts.

</details>

<details>
<summary><b>10. Scenario: orders appear out of sequence downstream. Why?</b></summary>

Probably no key or the wrong one, so events landed on different partitions, or retries reordered without idempotence. Use the order id as the key.

</details>

<details>
<summary><b>11. Scenario: a bad message makes the consumer crash in a loop. How do you handle it?</b></summary>

Catch the error, send the message to a dead-letter topic with its reason, and continue. Alert on the dead-letter rate.

</details>


---
# Check yourself before moving on

You're ready for the next notebook when you can tick these **without looking**:

- [ ] I can explain: Kafka is a log
- [ ] I can explain: Consumer groups, offsets and lag
- [ ] I can explain: Delivery guarantees in practice
- [ ] I can explain: Replication, retention and schemas

**Revision drill.** Come back in a few days, set a 25-minute timer per question, and solve these from a blank cell:

- [ ] [Q16](#q16) High Watermark
- [ ] [Q17](#q17) Rebalance Movement
- [ ] [Q18](#q18) Schema Compatibility
- [ ] [Q19](#q19) Kafka Disk Sizing
- [ ] [Q20](#q20) Outbox Pattern

Spaced repetition beats re-reading: redoing a question after you've partly forgotten it is what makes it stick.